# ML-07 — Baseline Action Score and Top-20 Review

## Purpose

This notebook turns an **observed, validated SEO signal** into a simple, auditable baseline action queue.

**Scope locked for this deliverable:** March 2026, `gsc_data_available IS TRUE`.

**Queue grain:** `(content_hash_id, report_date)` — intentionally preserved. Each row represents one content item on one report date, so the same content item may appear on multiple dates when the condition persists.

> **Decision-support only:** this baseline identifies pages worth human CTR/SERP review. It does not claim that a title, meta description, schema, or any other specific fix is the cause of zero clicks.


## 0. Evidence used to choose the rule

Before encoding a rule, I checked the actual March population rather than importing thresholds from the starter dataset.

The audit asked two separate questions:

1. **H1 — CTR vs. position:** does a real, sizeable segment exist where pages have strong visibility and page-one position but still receive zero clicks?
2. **H2 — volume + position headroom:** do high-impression pages just outside page one form a clearly supported quick-win segment?

H1 was judged **CONFIRMED**. H2 was judged **MIXED**.

Because H2 was not strong enough to support a scoring rule, the baseline is built only from the confirmed H1 mechanism.


### 0.1 Population sanity check

This query confirms the March `gsc_data_available IS TRUE` population and checks the known `gsc_avg_position = 0` placeholder.



In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_position_placeholder,
        SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")


**Observed result**

| n_total | n_position_placeholder | n_zero_impressions |
|---:|---:|---:|
| 3,611,061 | 163,189 | 0 |

**Interpretation:** there are 3.61M GSC-available rows in March. `gsc_avg_position = 0` is a placeholder and must not be treated as a real rank. There are no zero-impression rows in this available population.


### 0.2 Distribution check before choosing thresholds

The threshold was selected from this March population, not copied from the starter CSV.



In [ ]:
con.sql(f"""
    SELECT
        approx_quantile(gsc_impressions, [0.5, 0.75, 0.9, 0.95, 0.99]) AS impressions_pctiles,
        approx_quantile(
            gsc_clicks / NULLIF(gsc_impressions, 0),
            [0.5, 0.75, 0.9, 0.95, 0.99]
        ) AS ctr_pctiles,
        approx_quantile(gsc_avg_position, [0.5, 0.75, 0.9, 0.95, 0.99]) AS position_pctiles,
        COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position != 0
      AND gsc_impressions > 0
""")


**Observed result**

- Impressions P50/P75/P90/P95/P99 = **18 / 66 / 194 / 347 / 968**
- CTR P50/P75/P90/P95/P99 = **0 / 0 / 0.00356 / 0.01165 / 0.05033**
- Position P50/P75/P90/P95/P99 = **7.96 / 21.35 / 44.37 / 63.88 / 89.21**
- Rows used for the percentile calculation: **3,447,872**

**Locked threshold:** `194` impressions = observed **P90**.

**Locked position gate:** positions **1–10** = page one.

The threshold is therefore an empirical cutoff for this validated March slice, not a universal SEO rule.


### 0.3 Signal audit — position tier × impression bucket

CTR is calculated as a **weighted bucket CTR**:

`SUM(gsc_clicks) / SUM(gsc_impressions)`

This avoids the error of averaging row-level CTRs.


In [ ]:
con.sql(f"""
    WITH base AS (
        SELECT
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE
                WHEN gsc_avg_position BETWEEN 1 AND 10 THEN 'page_one'
                WHEN gsc_avg_position BETWEEN 11 AND 20 THEN 'headroom'
                ELSE 'rest'
            END AS position_tier,
            CASE
                WHEN gsc_impressions >= 194 THEN 'high_impressions'
                ELSE 'typical_impressions'
            END AS impression_bucket
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
          AND gsc_avg_position != 0
    )
    SELECT
        position_tier,
        impression_bucket,
        COUNT(*) AS n,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        ROUND(
            100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 3
        ) AS weighted_ctr_pct,
        CASE WHEN COUNT(*) < 50 THEN 'TOO SMALL (<50)' ELSE 'ok' END AS sample_flag
    FROM base
    GROUP BY position_tier, impression_bucket
    ORDER BY position_tier, impression_bucket
""")


**Observed result**

| Position tier | Impression bucket | n | Total impressions | Total clicks | Weighted CTR % |
|---|---|---:|---:|---:|---:|
| headroom | high | 27,029 | 11,527,129 | 39,950 | 0.347 |
| headroom | typical | 425,219 | 13,738,224 | 38,703 | 0.282 |
| page_one | high | 232,831 | 122,835,718 | 419,431 | 0.341 |
| page_one | typical | 1,685,916 | 61,942,240 | 219,056 | 0.354 |
| rest | high | 84,840 | 45,273,517 | 61,238 | 0.135 |
| rest | typical | 992,037 | 24,872,015 | 42,280 | 0.170 |

**H2 verdict: MIXED.** The high-impression headroom group exists and is large enough to inspect, but the evidence does not establish a clean enough "quick-win" mechanism to put it into the scoring rule.


### 0.4 Signal audit — within page-one CTR shape

The next check tests H1 directly: within page-one results, how much visibility is attached to rows receiving zero clicks?


In [ ]:
con.sql(f"""
    WITH base AS (
        SELECT
            gsc_impressions,
            gsc_clicks,
            gsc_clicks / gsc_impressions AS row_ctr,
            CASE
                WHEN gsc_impressions >= 194 THEN 'high_impressions'
                ELSE 'typical_impressions'
            END AS impression_bucket
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
          AND gsc_avg_position BETWEEN 1 AND 10
    )
    SELECT
        impression_bucket,
        COUNT(*) AS n,
        SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) AS n_zero_click,
        ROUND(
            100.0 * SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_zero_click,
        SUM(
            CASE WHEN gsc_clicks = 0 THEN gsc_impressions ELSE 0 END
        ) AS impressions_stuck_at_zero_click,
        approx_quantile(row_ctr, [0.25, 0.5, 0.75, 0.9]) AS row_ctr_pctiles
    FROM base
    GROUP BY impression_bucket
""")


**Observed result**

| Impression bucket | n | Zero-click rows | Zero-click % | Impressions stuck at zero clicks | CTR P25/P50/P75/P90 |
|---|---:|---:|---:|---:|---|
| typical | 1,685,916 | 1,518,843 | 90.09% | 47,718,384 | 0 / 0 / 0 / 0.00242 |
| high | 232,831 | 95,017 | 40.81% | 36,936,460 | 0 / 0.00216 / 0.00487 / 0.00911 |

**H1 verdict: CONFIRMED.**

The important observation is not merely that 40.81% is large. It is that, **within the same page-one tier**, there is a substantial high-visibility segment with zero clicks, carrying about **36.9M impressions**. The row-level CTR distribution also has real spread in the high-impression group.

This supports the existence of a reviewable "visibility not converting" segment. It does **not** prove a particular cause or guarantee future recovery.


## 1. My rule and its reason codes

### Rule

A page is flagged for CTR review when it:

1. ranks on **page one** (`gsc_avg_position` between 1 and 10),
2. has **meaningful visibility** (`gsc_impressions >= 194`, the observed March P90), and
3. received **zero clicks** (`gsc_clicks = 0`).

Among qualifying rows, **priority = raw `gsc_impressions`**. More impressions means more observed visibility is currently not converting.

### Why this rule

The signal audit confirmed the mechanism behind the rule. I am deliberately **not** adding a hand-picked position weight or the mixed H2 headroom signal to the score. That would introduce an unvalidated assumption.

### Rule outputs

- **Score:** `gsc_impressions`
- **Reason code:** `zero_click_high_visibility`
- **Action label:** `review_ctr`

`review_ctr` is intentionally an investigation action, not an automatic instruction to change the title/meta description/schema. The data identifies the symptom, not its causal fix.


### Rule preview

This preview shows the exact gate before writing the full queue.


In [ ]:
IMPRESSION_THRESHOLD = 194

preview = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        gsc_impressions AS score,
        'zero_click_high_visibility' AS reason_code,
        'review_ctr' AS action_label
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position BETWEEN 1 AND 10
      AND gsc_impressions >= {IMPRESSION_THRESHOLD}
      AND gsc_clicks = 0
    ORDER BY score DESC
    LIMIT 10
""")
preview


## 2. Build the ranked queue (writes the CSV)

The queue keeps the warehouse's natural daily grain: **`(content_hash_id, report_date)`**.

That is intentional. This deliverable is a **daily review queue**, not a deduplicated page-level table. Therefore, the same content item can legitimately appear on multiple dates when the qualifying condition persists.

The queue is restricted to the same March population used to derive the threshold, keeping the chain:

**validated population → observed distribution → threshold → rule → queue**


In [ ]:
import os

IMPRESSION_THRESHOLD = 194  # Observed P90 in March 2026

queue_df = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        gsc_impressions AS score,
        'zero_click_high_visibility' AS reason_code,
        'review_ctr' AS action_label
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position BETWEEN 1 AND 10
      AND gsc_impressions >= {IMPRESSION_THRESHOLD}
      AND gsc_clicks = 0
    ORDER BY score DESC
""").df()

print(f"Queue rows: {len(queue_df):,}")
print(f"Distinct content items: {queue_df['content_hash_id'].nunique():,}")
print(f"Score range: {queue_df['score'].min()} to {queue_df['score'].max()}")

os.makedirs("work/outputs", exist_ok=True)
queue_df.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")


**Observed queue result**

- **Queue rows:** 95,017
- **Distinct content items:** 16,578
- **Score range:** 194 → 37,368
- **Output:** `work/outputs/baseline_action_score.csv`

The difference between 95,017 rows and 16,578 distinct content items confirms that repeat appearances across dates are material. This is expected under the intentionally retained daily grain and is reviewed explicitly below.


## 3. Top-20 review

The top-20 review checks the **literal ranked artifact** that a human would receive. It therefore uses the exact top 20 rows of the queue without deduplication.

For each row:

- **Action** — what the specialist should do.
- **Reason code** — why the row entered the queue.
- **Confidence note** — how strongly the observed evidence supports the flag.
- **What would make it wrong** — a falsifiable condition that could make the review recommendation a false alarm.

The confidence here is **decision-support confidence**, not causal certainty.


In [ ]:
top20 = queue_df.head(20).reset_index(drop=True)

print(
    f"Top 20 rows represent "
    f"{top20['content_hash_id'].nunique()} distinct content items"
)
top20


**Observed top-20 diversity:** 15 distinct content items across 20 ranked rows.

### Hand review

Confidence is graded by how strong the position is (closer to position 1 = harder to
explain zero clicks by position alone), not by score. Score ranks by impressions only —
a deliberate, evidence-grounded choice (see Section 1) — so it will not always line up
with which row is the *most anomalous* case. Rows are cross-referenced where the same
content item reappears elsewhere in this top 20.

| Rank | Content item | Date | Position | Impressions | Action | Reason code | Confidence | What would make it wrong? |
|---:|---|---|---:|---:|---|---|---|---|
| 1 | `content_945d6ff91386c817` | 2026-03-04 | 8.61 | 37,368 | review_ctr | zero_click_high_visibility | Moderate | Weakest position in this top-20 (near the page-one boundary); if the page floats in and out of the top 10 across the month, "zero clicks in March" partly reflects inconsistent visibility rather than a stable CTR problem. |
| 2 | `content_34a70fea29d15f24` | 2026-03-22 | 3.13 | 27,410 | review_ctr | zero_click_high_visibility | High | If these impressions are concentrated in a short burst (e.g. one day) rather than spread across the month — only one date for this content item appears here, so a spike can't be ruled out. |
| 3 | `content_757b1fa67827358d` | 2026-03-13 | 2.26 | 19,301 | review_ctr | zero_click_high_visibility | High | If the aggregate position masks several underlying queries, most of them ranking well below 2.26 — the field is a monthly average, not a per-query breakdown. Second appearance of this item is row 11 (2026-03-29); compare before concluding this is a one-off. |
| 4 | `content_0c5606abaaab3178` | 2026-03-04 | 4.11 | 13,827 | review_ctr | zero_click_high_visibility | Moderate-High | If this is an isolated bad day. This same content item reappears at row 17 (2026-03-05, pos 4.29, also zero clicks) — two consecutive zero-click days argues *against* this being a one-off, strengthening rather than weakening the flag. |
| 5 | `content_046fc480045b88f5` | 2026-03-29 | 6.92 | 13,764 | review_ctr | zero_click_high_visibility | Moderate | If position 6.92 reflects large day-to-day rank volatility rather than a stable page-one placement — we only see the monthly average, not intraday movement. |
| 6 | `content_69379902126ff53f` | 2026-03-05 | 2.53 | 13,726 | review_ctr | zero_click_high_visibility | High | If these impressions are dominated by branded/navigational queries where users are known to scan without clicking through — no query-type field exists to check this. |
| 7 | `content_bf078007df823490` | 2026-03-29 | 1.04 | 13,253 | review_ctr | zero_click_high_visibility | Very high | Position ≈1 with zero clicks leaves little room to argue ranking is the issue. First of three appearances of this item in this top-20 (also rows 8, 10 — 2026-03-26/27/29); treat as one persistent case, not three independent ones (see Section 4). |
| 8 | `content_bf078007df823490` | 2026-03-27 | 1.06 | 11,952 | review_ctr | zero_click_high_visibility | Very high | Same basis as row 7. Second of three appearances (see rows 7, 10) — the recurrence across three separate dates argues against a one-day fluke. |
| 9 | `content_0bca6d9a85a9b408` | 2026-03-04 | 7.49 | 11,464 | review_ctr | zero_click_high_visibility | Moderate | Position sits close enough to the page-one/page-two boundary that the page may only be intermittently visible in the top 10 across the month, making the zero-click observation partly a visibility-consistency issue. |
| 10 | `content_bf078007df823490` | 2026-03-26 | 1.33 | 11,440 | review_ctr | zero_click_high_visibility | Very high | Third of three appearances of this item (see rows 7, 8) — three separate near-position-1, zero-click dates in the same week is the strongest case in this queue for a persistent condition, not noise. |
| 11 | `content_757b1fa67827358d` | 2026-03-29 | 3.10 | 9,719 | review_ctr | zero_click_high_visibility | High | Second appearance of this item (see row 3, 2026-03-13). If row 3's occurrence already triggered a review with no resolution yet, this row adds no new information — a queue-process gap, not a rule-quality problem. |
| 12 | `content_bb2a9972810ddd72` | 2026-03-22 | 4.88 | 9,566 | review_ctr | zero_click_high_visibility | Moderate-High | If a SERP feature (e.g. a featured snippet) is structurally pulling the answer onto the results page and suppressing clicks regardless of page quality — no SERP-feature field exists to check this. |
| 13 | `content_8e1334d6356668e3` | 2026-03-09 | 4.80 | 9,318 | review_ctr | zero_click_high_visibility | Moderate-High | If the audience for this query already knows the destination and is scanning for confirmation rather than intending to click — unverifiable without intent-level data. |
| 14 | `content_dc91779c3d085398` | 2026-03-29 | 2.34 | 8,977 | review_ctr | zero_click_high_visibility | High | If a large share of these impressions occurred in a short burst late in the window, leaving too little elapsed time for a click to register in this reporting period. Second appearance of this item is row 20 (2026-03-31); compare before concluding either date is representative. |
| 15 | `content_1ff6231687184dec` | 2026-03-04 | 6.67 | 8,923 | review_ctr | zero_click_high_visibility | Moderate | Position is borderline enough that this page may float near the edge of page one, making the zero-click pattern partly a visibility-consistency issue rather than a pure CTR issue. |
| 16 | `content_57487b3ef3d84b7d` | 2026-03-03 | 5.17 | 8,737 | review_ctr | zero_click_high_visibility | Moderate-High | If this single early-March date is not representative and the page converts normally on other dates — no other appearance of this content item exists in this top-20 to check against. |
| 17 | `content_0c5606abaaab3178` | 2026-03-05 | 4.29 | 8,649 | review_ctr | zero_click_high_visibility | Moderate-High | Second appearance of this item (see row 4, 2026-03-04, one day earlier, similar position). Two consecutive zero-click days strengthens the case for a persistent condition rather than weakening it. |
| 18 | `content_fa4cf3aa5ce67bb8` | 2026-03-30 | 1.70 | 8,481 | review_ctr | zero_click_high_visibility | Very high | If this is the single day this page briefly reached position ~1.7 and its typical March position was much worse — only one date for this content item appears in this top-20, so a temporary spike can't be ruled out. |
| 19 | `content_84c64d54522c71b0` | 2026-03-04 | 3.89 | 8,195 | review_ctr | zero_click_high_visibility | High | If the query mix behind these impressions is dominated by informational queries already answered directly on the SERP (e.g. a "People also ask" box) — unverifiable from the fields available here. |
| 20 | `content_dc91779c3d085398` | 2026-03-31 | 2.29 | 7,981 | review_ctr | zero_click_high_visibility | High | Same causal caveats as row 14 (its earlier appearance, 2026-03-29). Also notable: this row holds the strongest position in the bottom half of the queue (2.29) yet the *lowest* score in the top-20 — a visible instance of the score-vs-anomaly-strength limitation named in Section 4. |

**Review conclusion:** the top rows are internally consistent with the rule — every reviewed row has page-one position, at least 194 impressions, and zero clicks. Confidence tracks how close each row's position is to 1, not its score; the strongest-looking anomalies (rows 7, 8, 10, 18) are not the highest-scoring rows. Four content items repeat across the top 20 (rows 3/11, 4/17, 7/8/10, 14/20); in every repeated case the pattern is consistent rather than contradictory, which argues for persistence over noise — but see Section 4 for why that repetition is still a real queue-usability limitation.


## 4. Weak picks + leakage check

### Weak-pick finding

The main weakness exposed by the literal top-20 review is **repeat exposure of the same content item across different dates**.

Examples:

- `content_bf078007df823490` appears 3 times in the top 20.
- `content_757b1fa67827358d` appears 2 times.
- `content_0c5606abaaab3178` appears 2 times.
- `content_dc91779c3d085398` appears 2 times.

This is **not automatically a rule error**. A persistent zero-click/high-visibility condition can legitimately deserve repeated attention in a daily queue. However, it is a queue-usability limitation: a human reviewing the top 20 may see fewer than 20 unique pages.

A second limitation is that the available fields cannot distinguish a persistent CTR problem from a temporary/new-page condition, SERP-feature effect, or search-intent mismatch. Those are reasons to review the page, not reasons to claim a specific fix.

### Leakage check

The rule uses only same-window, pre-decision observed fields:

- `gsc_data_available`
- `gsc_avg_position`
- `gsc_impressions`
- `gsc_clicks`

It does **not** use:

- future dates,
- a future recovery label,
- shipped product flags such as `needs_ctr_fix` or `is_quick_win`,
- fitted model weights,
- or a future performance outcome.

Therefore this baseline is a **same-window rule-based decision-support baseline**, not a predictive recovery model.


### A second limitation: score tracks volume, not anomaly strength

The score (`gsc_impressions`) ranks qualifying rows by wasted audience size, not by how
strong the anomaly is. This is visible directly in the top 20: row 20
(`content_dc91779c3d085398`, position 2.29 — one of the strongest positions in the whole
queue) has the *lowest* score in the top 20, while row 1 (position 8.61 — the weakest
page-one position in the top 20) has the *highest*. Both priorities — "biggest wasted
audience" and "most anomalous case" — are legitimate ways to triage this list, but they
are not the same ordering, and the current score only serves the first one. This was a
deliberate choice (Section 1): the signal audit did not validate a position-weighting
relationship strongly enough to justify inventing one, so the score stays single-variable
rather than smuggling in an unvalidated weight. Worth naming as a known limitation, not a bug.


In [ ]:
# Structural leakage / grain sanity checks for the final queue

leakage_check = {
    "uses_future_dates": False,
    "uses_future_label": False,
    "uses_product_flags": False,
    "uses_fitted_weights": False,
    "queue_grain": "(content_hash_id, report_date)",
    "scope": "March 2026",
}

print(leakage_check)

# The queue should satisfy the rule by construction.
print("All queue rows page-one:", bool((queue_df["gsc_avg_position"].between(1, 10)).all()))
print("All queue rows >= threshold:", bool((queue_df["gsc_impressions"] >= IMPRESSION_THRESHOLD).all()))
print("All queue rows zero-click:", bool((queue_df["gsc_clicks"] == 0).all()))
print("All reason codes identical:", queue_df["reason_code"].nunique() == 1)
print("All action labels identical:", queue_df["action_label"].nunique() == 1)


## Final decision

### What this baseline does

> **Flag page-one content with ≥194 March impressions and zero clicks, then prioritize the flagged rows by impressions.**

- **Rule:** page one + ≥194 impressions + 0 clicks
- **Score:** `gsc_impressions`
- **Reason code:** `zero_click_high_visibility`
- **Action:** `review_ctr`
- **Grain:** `(content_hash_id, report_date)`
- **Scope:** March 2026
- **Queue size:** 95,017 rows
- **Distinct content items:** 16,578

### What it does not claim

It does not claim that the title, meta description, schema, search intent, or any other specific factor caused the zero clicks. Those are hypotheses for the human CTR/SERP review.

It also does not measure recovery performance yet. A future label and a proper evaluation metric are needed before claiming predictive value.

### Threshold caveat

`194` is the observed P90 for this March population. It should be treated as a **baseline calibration for this slice**, not as a universal SEO threshold.


## Self-check

- [x] Every section is filled with both reasoning and code.
- [x] The exploratory signal queries and their useful observed results are preserved.
- [x] The rule is based on a confirmed mechanism, not the mixed H2 signal.
- [x] The queue preserves the intended `(content_hash_id, report_date)` grain.
- [x] The top 20 reviews the literal ranked output, without hidden deduplication.
- [x] At least one weak-pick / limitation is explicitly documented.
- [x] No future window, future label, product flag, or fitted weight is used.
- [x] Claims use careful language: observed, measured, confirmed mechanism, decision-support.
- [ ] Run the notebook top-to-bottom in the project environment before committing.
- [ ] Confirm `work/outputs/baseline_action_score.csv` is present in the repo/output workflow required by the assignment.
